# Relational Tables and Shared Data

*Store shared facts once and use a join to connect related records.*

SQL (Structured Query Language) expresses operations on database tables.
SQLite is the database engine used here, and Python's built-in `sqlite3`
module sends SQL statements to that engine without a separate server.


## Tables and Shared Facts

A relational database stores records as **rows** and named fields as
**columns**. The tables below resemble DataFrames, but their keys also
describe how stored records relate to one another.

Students can share a department, while each department has one building
in this example. Keeping the building in a separate department table
avoids repeating the same fact in every student record.

| Table | One row represents | Key |
|---|---|---|
| `departments` | A department and its building | `major` |
| `students` | A student and their major | `student_id` |

A **primary key** uniquely identifies a row. A **foreign key** refers
to a key in another table: `students.major` refers to `departments.major`.
Multiple students can reference the same department.


## Create Two Related Tables

`sqlite3.connect(":memory:")` opens a database that lasts for this
connection. `connection.execute()` runs one SQL statement. The Python
triple quotes hold a multiline string; the text inside is SQL.
Uppercase SQL keywords make them easy to distinguish from table names.

`CREATE TABLE` declares columns; `TEXT` and `INTEGER` describe the text
and integer fields used here. `NOT NULL` requires a value.
`student_id INTEGER PRIMARY KEY` lets SQLite assign an ID when a new
student row omits it.

`REFERENCES` declares the relationship. SQLite's
`PRAGMA foreign_keys = ON` enables its enforcement on this connection
before inserting rows, so a student must refer to an existing department.


In [1]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")
connection.execute("""
CREATE TABLE departments (
    major TEXT PRIMARY KEY NOT NULL,
    building TEXT NOT NULL
)
""")
connection.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    major TEXT NOT NULL REFERENCES departments(major)
)
""")
print("Created departments and students tables.")


Created departments and students tables.


## Insert the Prepared Records

`INSERT INTO ... VALUES (...)` adds a row. `executemany()` repeats that
statement for the supplied tuples, binding Python values to `?`
placeholders in order. Insert departments first so the student references
already exist. `commit()` completes the data-changing transaction.

`SELECT ... FROM ...` reads columns. `ORDER BY` gives the result a
defined order. Executing a query returns a cursor, which the `for` loop
reads one row at a time. Each selected row is a Python tuple.


In [2]:
department_rows = [("CS", "Engineering Hall"), ("Math", "Science Hall")]
student_rows = [("Mina", "CS"), ("Jisoo", "Math"), ("Kevin", "CS")]

connection.executemany(
    "INSERT INTO departments (major, building) VALUES (?, ?)", department_rows
)
connection.executemany(
    "INSERT INTO students (student_name, major) VALUES (?, ?)", student_rows
)
connection.commit()

print("Students:")
for row in connection.execute("SELECT student_name, major FROM students ORDER BY student_id"):
    print(row)
print("Departments:")
for row in connection.execute("SELECT major, building FROM departments ORDER BY major"):
    print(row)


Students:
('Mina', 'CS')
('Jisoo', 'Math')
('Kevin', 'CS')
Departments:
('CS', 'Engineering Hall')
('Math', 'Science Hall')


Mina and Kevin both refer to CS. Engineering Hall appears once in the department table instead of being stored separately for each student.


## Join on the Shared Key

`JOIN ... ON ...` matches related records. The aliases `s` and `d`
shorten the table names; `s.major` identifies the column in `students`.
This inner join returns students with matching departments. Each department
key is unique, so every student matches exactly one department here.


In [3]:
joined_query = """
SELECT s.student_name, s.major, d.building
FROM students AS s
JOIN departments AS d ON s.major = d.major
ORDER BY s.student_id
"""

print("Student, major, and building:")
for row in connection.execute(joined_query):
    print(row)


Student, major, and building:
('Mina', 'CS', 'Engineering Hall')
('Jisoo', 'Math', 'Science Hall')
('Kevin', 'CS', 'Engineering Hall')


Mina and Kevin receive Engineering Hall, while Jisoo receives Science Hall. The join reconstructs the combined view from the two stored tables.


## Update a Shared Fact Once

`UPDATE ... SET ... WHERE ...` changes only the rows matching the condition.
The placeholders bind the new building and the department key.


In [4]:
connection.execute(
    "UPDATE departments SET building = ? WHERE major = ?", ("AI Center", "CS")
)
connection.commit()

print("Joined result after one department update:")
for row in connection.execute(joined_query):
    print(row)


Joined result after one department update:
('Mina', 'CS', 'AI Center')
('Jisoo', 'Math', 'Science Hall')
('Kevin', 'CS', 'AI Center')


Two students, Mina and Kevin, now show AI Center. Jisoo still shows Science Hall. The student rows did not need separate building updates.


## Close the Connection

`.close()` releases the database connection. This database lives only
in memory, so its tables disappear when the connection closes.


In [5]:
connection.close()
print("Connection closed.")


Connection closed.


Store a shared fact in one table and connect related records with keys. A join can recover a combined view while one update changes the shared fact.

[SQLite foreign keys](https://www.sqlite.org/foreignkeys.html)
